[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssuai/Hands-On-Large-Language-Models/blob/main/chapter02/lab_2_2_embeddings.ipynb)

### [OPTIONAL] - Installing Packages on Colab

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [2]:
%%capture
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.3 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1

# Contextualized Word Embeddings From a Language Model (Like BERT)

In [3]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
output.shape

torch.Size([1, 4, 384])

In [5]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


In [6]:
output

tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)

# Text Embeddings (For Sentences and Whole Documents)

In [21]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
vector.shape

(768,)

In [20]:
vector

array([-2.02203933e-02,  4.57696877e-02, -1.26637146e-02, -3.37991631e-03,
       -2.54910300e-03, -3.31096235e-03, -4.58366945e-02,  2.32390799e-02,
       -3.12585123e-02, -3.15588824e-02, -2.19783355e-02,  3.67821753e-03,
        3.39105981e-03, -1.88130587e-02, -2.65821870e-02, -1.45892315e-02,
        3.20066400e-02, -3.66493082e-03,  1.75410733e-02,  5.43209612e-02,
       -3.96661647e-02,  9.93637647e-03, -3.27205285e-02, -1.62909944e-02,
        6.36525359e-03, -1.24459760e-02,  1.04737049e-02,  3.08674313e-02,
        7.47453189e-03, -3.86319868e-02, -6.19924963e-02, -3.25308628e-02,
       -1.40268244e-02,  5.16586639e-02,  1.69651707e-06, -1.74092929e-04,
       -1.27881172e-03, -4.15052772e-02,  5.00347884e-03,  4.58600894e-02,
       -2.85337027e-02,  5.02370670e-02, -1.75183658e-02, -3.98830482e-04,
        1.63311344e-02,  5.89936748e-02, -2.21730098e-02, -4.00975086e-02,
       -6.60439581e-03, -2.97632981e-02, -3.18221636e-02, -1.77945811e-02,
       -1.54363075e-02, -

In [23]:
vector2 = model.encode("I watched the movie One Battle After Another last weekend. It was the best movie ever!")
vector2

array([-2.67506428e-02,  5.06745875e-02,  1.52687719e-02,  3.46124731e-02,
        2.63262745e-02, -1.36933671e-02, -4.17678170e-02,  5.22871222e-03,
       -7.65343234e-02, -1.31951850e-02, -7.50181377e-02, -6.53573358e-03,
       -1.66906056e-03, -2.04616580e-02,  2.87406668e-02,  2.48090131e-04,
        5.69002144e-02, -2.56392229e-02, -5.40490709e-02,  3.78300026e-02,
        1.43145369e-02, -4.81950343e-02,  3.43623571e-03, -4.29103710e-02,
       -2.04771403e-02,  3.15443762e-02, -1.72283780e-02, -1.39464475e-02,
        6.31325543e-02, -9.94180739e-02,  3.74041200e-02, -8.08880925e-02,
        2.37063672e-02,  8.99729207e-02,  1.04875960e-06, -4.53571491e-02,
        5.67712472e-04, -8.82114936e-03,  1.50912302e-02,  5.11959307e-02,
       -6.14833739e-03,  4.49514575e-02, -6.04041815e-02, -8.00653771e-02,
        3.92423831e-02,  1.44530507e-02,  1.12929279e-02,  6.08630069e-02,
       -2.11280398e-03, -5.59169836e-02, -4.00140276e-03, -2.84913313e-02,
        4.86506820e-02,  

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

# Reshape vectors to be 2D arrays as cosine_similarity expects 2D input
vector_reshaped = vector.reshape(1, -1)
vector2_reshaped = vector2.reshape(1, -1)

# Calculate cosine similarity
similarity = cosine_similarity(vector_reshaped, vector2_reshaped)[0][0]

print(f"Similarity between vector and vector2: {similarity}")

Similarity between vector and vector2: 0.9252984523773193


# Word Embeddings Beyond LLMs


In [9]:
!pip install gensim==4.3.3

In [10]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [11]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

# Recommending songs by embeddings

In [12]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [13]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [14]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [15]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('5634', 0.9985853433609009),
 ('2849', 0.9981250762939453),
 ('6626', 0.9966500401496887),
 ('2014', 0.9965460300445557),
 ('1922', 0.9959208369255066),
 ('3167', 0.9958561062812805),
 ('2976', 0.9956579208374023),
 ('6624', 0.9954155087471008),
 ('5586', 0.9953592419624329),
 ('11473', 0.9948761463165283)]

In [16]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [17]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
5634,Mr. Brownstone,Guns N' Roses
2849,Run To The Hills,Iron Maiden
6626,Blackout,Scorpions
2014,Youth Gone Wild,Skid Row
1922,One,Metallica


In [18]:
print_recommendations(2172)

,title,artist
id,,
5634,Mr. Brownstone,Guns N' Roses
2849,Run To The Hills,Iron Maiden
6626,Blackout,Scorpions
2014,Youth Gone Wild,Skid Row
1922,One,Metallica


In [19]:
print_recommendations(842)

,title,artist
id,,
890,Knock You Down (w\/ Ne-Yo & Kanye West),Keri Hilson
886,Heartless,Kanye West
453,Temperature,Sean Paul
11331,Take You There,Sean Kingston
5698,Turnin' Me On (w\/ Lil Wayne),Keri Hilson
